# AIFS vs IFS Comparison — Exceedance Probability Skill

This notebook evaluates whether ECMWF's AI-based forecast system (AIFS)
produces exceedance probabilities with better or worse skill than the
traditional Integrated Forecasting System (IFS) for East Africa flood events.

**Method**: For each historical event in the EM-DAT catalogue, we extract the
exceedance probability issued T+1 to T+7 days before the event onset for the
affected admin-1 unit. We compare Brier Scores, reliability diagrams, and
AUC-ROC across both model sources.

> **Note**: This notebook requires pre-computed exceedance Zarr stores for both
> IFS and AIFS. Populate the paths below before running.

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# --- Configure paths ---
IFS_EXCEEDANCE_URI  = 'data/exceedance_ifs.zarr'   # update to actual URI
AIFS_EXCEEDANCE_URI = 'data/exceedance_aifs.zarr'  # update to actual URI
EMDAT_CSV           = 'data/emdat_east_africa.csv'  # update to actual path

LEAD_DAYS     = [1, 3, 5, 7]
WINDOW_H      = 24
RETURN_PERIOD = 5

## 5.1  Load Exceedance Stores

In [ ]:
def _open_store(uri: str) -> xr.Dataset | None:
    try:
        return xr.open_zarr(uri, consolidated=False)
    except Exception as e:
        print(f'Could not open {uri}: {e}')
        return None

ifs_ds  = _open_store(IFS_EXCEEDANCE_URI)
aifs_ds = _open_store(AIFS_EXCEEDANCE_URI)

if ifs_ds is not None:
    print('IFS store:', ifs_ds)
if aifs_ds is not None:
    print('AIFS store:', aifs_ds)

## 5.2  Load EM-DAT Flood Events

In [ ]:
def _load_emdat(path: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(path)
        df['start_date'] = pd.to_datetime(df['start_date'])
        return df
    except FileNotFoundError:
        print(f'EM-DAT file not found: {path}')
        return pd.DataFrame()

emdat = _load_emdat(EMDAT_CSV)
if not emdat.empty:
    print(f'Loaded {len(emdat)} EM-DAT events')
    print(emdat[['country', 'admin1_pcode', 'start_date']].head())

## 5.3  Brier Score by Lead Time

In [ ]:
def brier_score(forecast: np.ndarray, observed: np.ndarray) -> float:
    return float(np.mean((forecast - observed) ** 2))

def extract_exceedance_at_event(
    ds: xr.Dataset,
    event_date: pd.Timestamp,
    admin_lat: float,
    admin_lon: float,
    lead_days: int,
) -> float | None:
    forecast_date = event_date - pd.Timedelta(days=lead_days)
    try:
        subset = ds['exceedance_prob'].sel(
            date=forecast_date,
            window=WINDOW_H,
            return_period=RETURN_PERIOD,
            method='nearest',
        )
        return float(subset.sel(latitude=admin_lat, longitude=admin_lon, method='nearest'))
    except Exception:
        return None

if not emdat.empty and ifs_ds is not None:
    # placeholder: in production, join emdat with admin centroid coordinates
    print('Brier score computation requires event-admin spatial join — see section 5.4.')
else:
    print('No data loaded — configure paths above to run comparison.')

## 5.4  Reliability Diagram (placeholder)

A reliability diagram bins forecast probabilities and plots the
observed frequency of flood occurrence in each bin.
Perfect reliability lies on the diagonal.

In [ ]:
import matplotlib.pyplot as plt

def plot_reliability(forecast_probs, observed_labels, model_name='Model', ax=None):
    bins = np.linspace(0, 1, 11)
    bin_centres, obs_freq = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (forecast_probs >= lo) & (forecast_probs < hi)
        if mask.sum() > 0:
            bin_centres.append((lo + hi) / 2)
            obs_freq.append(float(observed_labels[mask].mean()))
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect reliability')
    ax.plot(bin_centres, obs_freq, 'o-', label=model_name)
    ax.set_xlabel('Forecast probability')
    ax.set_ylabel('Observed frequency')
    ax.set_title(f'Reliability diagram — {model_name}')
    ax.legend()
    return ax

# Example with synthetic data (replace with real arrays)
rng = np.random.default_rng(0)
syn_probs    = rng.uniform(0, 1, 500)
syn_observed = (rng.uniform(0, 1, 500) < syn_probs * 0.7 + 0.1).astype(float)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), tight_layout=True)
plot_reliability(syn_probs, syn_observed, 'IFS (synthetic)', axes[0])
plot_reliability(syn_probs * 0.9 + 0.05, syn_observed, 'AIFS (synthetic)', axes[1])
plt.suptitle('Replace with real IFS / AIFS data')
plt.show()

---
Populate `IFS_EXCEEDANCE_URI`, `AIFS_EXCEEDANCE_URI`, and `EMDAT_CSV` with
production paths to run the full comparison.